# C10 — Model Explainability: GradCAM, Integrated Gradients, and SHAP

> **Audience**: PhD students · **Framework**: PyTorch + Captum · **Model**: ResNet-50

Explainability methods answer: *which input features drove this prediction?*

This is critical for:
- **Debugging**: catching spurious correlations (e.g., a model that classifies grass as "cow")
- **Trust and deployment**: regulatory requirements (GDPR Article 22, FDA medical device guidelines)
- **Scientific discovery**: using models as tools to generate hypotheses about data

**Methods covered** (ordered by complexity and faithfulness)

| Method | Type | Requires retraining? | Faithfulness |
|---|---|---|---|
| Vanilla Saliency | Gradient | No | Low |
| GradCAM | Gradient × Activation | No | Medium |
| GradCAM++ | Weighted GradCAM | No | Medium-High |
| Integrated Gradients | Path integral | No | High (axioms) |
| SHAP (KernelSHAP) | Game theory | No | Highest (Shapley)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
import requests
from io import BytesIO

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print(f"Device: {DEVICE}")

In [ ]:
# ── Load pretrained ResNet-50 and a sample image ──────────────────────────────
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.eval().to(DEVICE)

# Standard ImageNet preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Helper: denormalise for display
def denorm(tensor: torch.Tensor) -> np.ndarray:
    """Converts a normalised ImageNet tensor to a displayable uint8 numpy array."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img  = (tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    return (img * 255).astype(np.uint8)

# Load a sample image (dog + cat on a sofa — multiple objects for interesting attribution)
url   = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Dog_Breeds.jpg/640px-Dog_Breeds.jpg"
try:
    resp  = requests.get(url, timeout=10)
    image = Image.open(BytesIO(resp.content)).convert("RGB")
except Exception:
    # Fallback: generate a synthetic image
    image = Image.fromarray(np.random.randint(100, 200, (224, 224, 3), dtype=np.uint8))

# (1, 3, 224, 224)
input_tensor = preprocess(image).unsqueeze(0).to(DEVICE)
input_tensor.requires_grad_(True)

# Get top prediction
with torch.no_grad():
    logits = model(input_tensor)
    probs  = F.softmax(logits, dim=1)
    top5   = probs[0].topk(5)

# Load ImageNet class labels
imagenet_labels = {i: v for i, v in enumerate(models.ResNet50_Weights.DEFAULT.meta["categories"])}
pred_class_idx  = top5.indices[0].item()
pred_class_name = imagenet_labels[pred_class_idx]

print(f"Top prediction: [{pred_class_idx}] {pred_class_name}  (prob={top5.values[0].item():.4f})")
print("Top-5:")
for idx, val in zip(top5.indices.tolist(), top5.values.tolist()):
    print(f"  [{idx:4d}] {imagenet_labels[idx]:<40s} {val:.4f}")

plt.figure(figsize=(4,4))
plt.imshow(image.resize((224,224))); plt.axis("off")
plt.title(f"Input: {pred_class_name}"); plt.tight_layout(); plt.show()

# 1) Vanilla Saliency Map

**The simplest attribution method** (Simonyan et al., 2014)

The gradient of the class score with respect to the input pixels tells us:
*which pixels, if slightly changed, would most affect the prediction?*

$$S_{ij} = \left| \frac{\partial y_c}{\partial x_{ij}} \right|$$

where $y_c$ is the logit for class $c$ and $x_{ij}$ is pixel $(i,j)$.

**Limitations**
- Sensitive to gradient saturation (neurons can be at saturation → gradient ≈ 0)
- Noisy, pixel-level patterns that are hard to interpret
- Does not satisfy the completeness axiom (gradients don't sum to the output)

In [ ]:
def vanilla_saliency(
    model: nn.Module,
    input_tensor: torch.Tensor,
    class_idx: int,
) -> np.ndarray:
    """
    Computes vanilla saliency map: |d(logit_c)/d(input)|.

    Args:
        model        : Pretrained classifier
        input_tensor : (1, 3, H, W) preprocessed input; must have requires_grad=True
        class_idx    : Target class index

    Returns:
        saliency : (H, W) saliency map, values in [0, 1]
    """
    model.eval()

    # Enable gradient tracking on input
    x = input_tensor.clone().requires_grad_(True)

    # Forward pass — we want the gradient of the class logit w.r.t. input
    # (1, 3, H, W) → (1, num_classes)
    logits = model(x)

    # Zero all gradients, then backprop through the target class score only
    model.zero_grad()
    # Scalar → backprop; selects gradient for class_idx
    logits[0, class_idx].backward()

    # |d logit_c / d x|: take absolute value then max over channels
    # (1, 3, H, W) → (H, W)
    saliency = x.grad.data.abs().max(dim=1)[0].squeeze().cpu().numpy()

    # Normalise to [0, 1] for display
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)

    return saliency


saliency_map = vanilla_saliency(model, input_tensor, pred_class_idx)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(denorm(input_tensor[0])); axes[0].set_title("Input"); axes[0].axis("off")
axes[1].imshow(saliency_map, cmap="hot"); axes[1].set_title("Vanilla Saliency"); axes[1].axis("off")
plt.suptitle(f"Target class: {pred_class_name}")
plt.tight_layout(); plt.show()

# 2) GradCAM — Gradient-weighted Class Activation Mapping

**Why GradCAM over saliency?** (Selvaraju et al., 2017)

Saliency maps are pixel-level — noisy and hard to interpret. GradCAM uses the
*last convolutional layer's feature maps*, producing a coarser but semantically
meaningful localisation map.

**The algorithm**

1. Forward pass → get the last conv feature maps $A^k$ (shape: $H' \times W'$)
2. Backprop → get gradients $\frac{\partial y_c}{\partial A^k}$
3. Global average pool the gradients to get importance weights:

$$\alpha_k^c = \frac{1}{H'W'} \sum_{i,j} \frac{\partial y_c}{\partial A^k_{ij}}$$

4. Weighted sum of feature maps, then ReLU:

$$L^c_{GradCAM} = \text{ReLU}\!\left( \sum_k \alpha_k^c A^k \right)$$

**Why ReLU?**
We only care about features that *positively* contribute to the target class.
Negative contributions would be suppressed by the model's own activations.

In [ ]:
class GradCAM:
    """
    GradCAM: computes gradient-weighted class activation maps.

    Attaches forward and backward hooks to a target layer,
    capturing both activations and gradients during a forward-backward pass.
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module) -> None:
        self.model   = model
        self.activations: torch.Tensor = None
        self.gradients:   torch.Tensor = None

        # Register hooks to capture activations and gradients at target_layer
        self._fwd_hook = target_layer.register_forward_hook(self._save_activation)
        self._bwd_hook = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output) -> None:
        # output: (batch_num, n_channels, H', W') — feature map after target layer
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output) -> None:
        # grad_output[0]: (batch_num, n_channels, H', W') — gradients w.r.t. feature map
        self.gradients = grad_output[0].detach()

    def __call__(
        self,
        input_tensor: torch.Tensor,
        class_idx: int,
    ) -> np.ndarray:
        """
        Computes GradCAM for the specified class.

        Args:
            input_tensor : (1, 3, H, W) preprocessed input
            class_idx    : Target class index

        Returns:
            cam : (H, W) GradCAM heatmap normalised to [0, 1]
        """
        self.model.eval()
        x = input_tensor.clone().requires_grad_(True)

        # Forward pass — hooks capture activations
        # (1, 3, H, W) → (1, num_classes)
        logits = self.model(x)

        self.model.zero_grad()
        # Backprop through target class — hooks capture gradients
        logits[0, class_idx].backward()

        # alpha_k: global average pool of gradients over spatial dims
        # (1, n_channels, H', W') → (1, n_channels, 1, 1)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)

        # Weighted sum of activations
        # (1, n_channels, H', W') * (1, n_channels, 1, 1) → (1, n_channels, H', W')
        # → sum over channels → (1, H', W')
        cam = (weights * self.activations).sum(dim=1, keepdim=True)

        # ReLU: only positive contributions matter for the target class
        # (1, 1, H', W')
        cam = F.relu(cam)

        # Upsample to input resolution
        # (1, 1, H', W') → (1, 1, H, W)
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        # Normalise to [0, 1]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def remove_hooks(self) -> None:
        self._fwd_hook.remove()
        self._bwd_hook.remove()


def overlay_cam(image_rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    """Overlays a CAM heatmap on a PIL image for visualisation."""
    heatmap = cm.jet(cam)[..., :3]  # (H, W, 3) RGB heatmap
    heatmap = (heatmap * 255).astype(np.uint8)
    overlay = (alpha * image_rgb + (1 - alpha) * heatmap).astype(np.uint8)
    return overlay


# Hook the last conv block of ResNet-50 (layer4[-1].conv3)
target_layer = model.layer4[-1].conv3
gradcam      = GradCAM(model, target_layer)

cam_map = gradcam(input_tensor, pred_class_idx)

img_array = np.array(image.resize((224, 224)))
overlay   = overlay_cam(img_array, cam_map)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(img_array); axes[0].set_title("Input"); axes[0].axis("off")
axes[1].imshow(cam_map, cmap="jet"); axes[1].set_title("GradCAM"); axes[1].axis("off")
axes[2].imshow(overlay); axes[2].set_title("Overlay"); axes[2].axis("off")
plt.suptitle(f"GradCAM — target class: {pred_class_name}")
plt.tight_layout(); plt.show()
gradcam.remove_hooks()

# 3) GradCAM++

**What GradCAM++ improves** (Chattopadhay et al., 2018)

GradCAM averages gradients globally. This treats all spatial locations equally —
it may miss objects that appear *multiple times* in the image or appear in
different locations at different scales.

GradCAM++ weights each gradient value individually using a
*pixel-wise importance weight* derived from the second-order gradient:

$$\alpha_{ij}^{kc} = \frac{\frac{\partial^2 y_c}{\partial (A^k_{ij})^2}}{2 \frac{\partial^2 y_c}{\partial (A^k_{ij})^2} + \sum_{a,b} A^k_{ab} \frac{\partial^3 y_c}{\partial (A^k_{ij})^3}}$$

This pixel-wise weighting focuses the CAM on the specific activations
that contributed most to the final score, rather than spreading importance
evenly across the spatial dimensions.

In [ ]:
class GradCAMPlusPlus:
    """
    GradCAM++: pixel-wise importance weighting of gradients.

    More accurate for multiple object instances and partial occlusions.
    Uses the same hook-based approach as GradCAM.
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module) -> None:
        self.model        = model
        self.activations  = None
        self.gradients    = None
        self._fwd_hook    = target_layer.register_forward_hook(self._save_activation)
        self._bwd_hook    = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, m, i, o): self.activations = o.detach()
    def _save_gradient(self, m, gi, go): self.gradients   = go[0].detach()

    def __call__(self, input_tensor: torch.Tensor, class_idx: int) -> np.ndarray:
        """
        Computes GradCAM++ for the specified class.

        Args:
            input_tensor : (1, 3, H, W)
            class_idx    : Target class index

        Returns:
            cam : (H, W) normalised GradCAM++ heatmap
        """
        self.model.eval()
        x = input_tensor.clone().requires_grad_(True)

        # Forward pass
        # (1, 3, H, W) → (1, num_classes)
        logits = self.model(x)
        self.model.zero_grad()
        logits[0, class_idx].backward()

        # Activations: (1, n_channels, H', W')
        A  = self.activations
        # Gradients: (1, n_channels, H', W')
        dY = self.gradients

        # Compute alpha weights for GradCAM++ (numerically stable version)
        # Numerator: gradient squared
        # (1, n_channels, H', W')
        grad_sq  = dY ** 2
        # Denominator: 2 * grad^2 + sum_ab(A_ab * grad^3)
        # sum over spatial dims for each channel: (1, n_channels, 1, 1)
        sum_A_grad3 = (A * dY ** 3).sum(dim=(2, 3), keepdim=True)
        # (1, n_channels, H', W')
        alpha    = grad_sq / (2 * grad_sq + sum_A_grad3 + 1e-8)

        # Only keep positive gradients (equivalent to applying ReLU on the gradient)
        # (1, n_channels, H', W')
        alpha    = alpha * F.relu(dY)

        # Pixel-wise importance weights → channel weights via spatial average
        # (1, n_channels, H', W') → (1, n_channels, 1, 1)
        weights  = alpha.mean(dim=(2, 3), keepdim=True)

        # Weighted sum of feature maps, ReLU, upsample
        # (1, n_channels, H', W') → (H, W)
        cam = F.relu((weights * A).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def remove_hooks(self): self._fwd_hook.remove(); self._bwd_hook.remove()


target_layer_pp   = model.layer4[-1].conv3
gradcam_pp        = GradCAMPlusPlus(model, target_layer_pp)
cam_pp            = gradcam_pp(input_tensor, pred_class_idx)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(img_array);    axes[0].set_title("Input");    axes[0].axis("off")
axes[1].imshow(overlay_cam(img_array, cam_map), ); axes[1].set_title("GradCAM");    axes[1].axis("off")
axes[2].imshow(overlay_cam(img_array, cam_pp));    axes[2].set_title("GradCAM++");  axes[2].axis("off")
plt.suptitle(f"GradCAM vs GradCAM++ — {pred_class_name}")
plt.tight_layout(); plt.show()
gradcam_pp.remove_hooks()

# 4) Integrated Gradients

**The problem with gradient-based methods**
Vanilla saliency and GradCAM both use local gradients, which can be misleading:
- A feature saturated to its maximum contribution will have gradient ≈ 0 even though it
  is highly important (the *saturation problem*)
- Gradients do not satisfy the *completeness axiom*

**Integrated Gradients** (Sundararajan et al., 2017) solves this by integrating
the gradient along a straight-line path from a baseline $x'$ (usually the zero image)
to the actual input $x$:

$$IG_i(x) = (x_i - x'_i) \int_0^1 \frac{\partial F(x' + \alpha(x-x'))}{\partial x_i} d\alpha$$

Approximated numerically with Riemann summation over `n_steps` interpolations.

**Completeness axiom** (key advantage)
$$\sum_i IG_i(x) = F(x) - F(x')$$

The sum of attributions exactly equals the difference in model output between
the input and the baseline. This provides a strong sanity check.

In [ ]:
def integrated_gradients(
    model: nn.Module,
    input_tensor: torch.Tensor,
    class_idx: int,
    baseline: torch.Tensor = None,
    n_steps: int = 50,
) -> np.ndarray:
    """
    Computes Integrated Gradients attribution for a target class.

    Integrates gradients along the straight-line path from baseline to input
    using Riemann approximation with n_steps interpolations.

    Args:
        model        : Pretrained classifier (no modification needed)
        input_tensor : (1, 3, H, W) preprocessed input
        class_idx    : Target class index to explain
        baseline     : (1, 3, H, W) reference point (black image if None)
        n_steps      : Number of interpolation steps (higher → more accurate)

    Returns:
        attribution : (H, W) attribution map, normalised to [0, 1]
    """
    model.eval()

    if baseline is None:
        # Black image baseline — a natural "absence of information" reference
        # (1, 3, H, W)
        baseline = torch.zeros_like(input_tensor)

    # Build n_steps interpolations between baseline and input
    # (n_steps, 3, H, W)
    alphas      = torch.linspace(0, 1, n_steps, device=input_tensor.device)
    # (n_steps, 1, 1, 1) for broadcasting
    alphas      = alphas.view(-1, 1, 1, 1)
    # Interpolated inputs: x' + alpha*(x - x')
    # (n_steps, 3, H, W)
    interpolated = baseline + alphas * (input_tensor - baseline)
    interpolated = interpolated.requires_grad_(True)

    # Forward pass for all interpolated inputs
    # (n_steps, 3, H, W) → (n_steps, num_classes)
    logits = model(interpolated)

    # Backprop through the target class score for all steps simultaneously
    model.zero_grad()
    # Sum over batch to get a scalar, then backprop
    logits[:, class_idx].sum().backward()

    # Gradients at each interpolation point
    # (n_steps, 3, H, W)
    grads = interpolated.grad.detach()

    # Riemann approximation of the integral: average gradients
    # (n_steps, 3, H, W) → (3, H, W)
    avg_grads = grads.mean(dim=0)

    # Multiply by (input - baseline) — the path direction
    # (3, H, W) * (3, H, W) → (3, H, W)
    integrated_grads = avg_grads * (input_tensor - baseline).squeeze(0).detach()

    # Sum over channel dimension for a single attribution map
    # (3, H, W) → (H, W)
    attribution = integrated_grads.abs().sum(dim=0).cpu().numpy()

    # Normalise
    attribution = (attribution - attribution.min()) / (attribution.max() - attribution.min() + 1e-8)
    return attribution


ig_map = integrated_gradients(model, input_tensor, pred_class_idx, n_steps=50)

# Completeness check: sum(IG) should ≈ F(input) - F(baseline)
with torch.no_grad():
    f_input    = model(input_tensor)[0, pred_class_idx].item()
    f_baseline = model(torch.zeros_like(input_tensor))[0, pred_class_idx].item()
delta_f = f_input - f_baseline

# The raw (unsigned) integrated grads sum
ig_raw = integrated_gradients.__wrapped__ if hasattr(integrated_gradients, "__wrapped__") else None
print(f"F(input) - F(baseline) = {delta_f:.4f}  (completeness denominator)")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(img_array);                         axes[0].set_title("Input");               axes[0].axis("off")
axes[1].imshow(overlay_cam(img_array, cam_map));   axes[1].set_title("GradCAM");             axes[1].axis("off")
axes[2].imshow(overlay_cam(img_array, ig_map));    axes[2].set_title("Integrated Gradients"); axes[2].axis("off")
plt.suptitle(f"Attribution comparison — {pred_class_name}")
plt.tight_layout(); plt.show()

# 5) Captum — Comprehensive Attribution Library

**Captum** (Facebook AI, 2020) provides production-grade, numerically stable
implementations of all major attribution methods, including:

- Integrated Gradients (same as Section 4, but with more efficient batching)
- DeepLIFT (backpropagation-based, faster than IG)
- SHAP (KernelSHAP, GradientSHAP)
- Occlusion
- Noise Tunnel (SmoothGrad — reduces gradient noise via averaging)
- Feature Ablation

**When to use Captum vs rolling your own**
For research papers: roll your own for the primary method (shows understanding),
use Captum for comparison baselines and ablations.

In [ ]:
# Install: !pip install captum --quiet

from captum.attr import IntegratedGradients, GradientShap, Occlusion, NoiseTunnel
from captum.attr import visualization as viz

# ── Integrated Gradients (Captum) ─────────────────────────────────────────────
ig_captum   = IntegratedGradients(model)
# (1, 3, H, W) → (1, 3, H, W) attributions
ig_attr     = ig_captum.attribute(
    input_tensor,
    target       = pred_class_idx,
    n_steps      = 50,
    internal_batch_size = 25,  # efficient chunked forward passes
)

# Visualise with Captum's utility
fig, axes = viz.visualize_image_attr_multiple(
    ig_attr[0].permute(1, 2, 0).cpu().detach().numpy(),
    denorm(input_tensor[0]),
    ["original_image", "heat_map", "masked_image"],
    ["all", "absolute_value", "absolute_value"],
    ["Input", "IG Attribution", "Masked Input"],
    show_colorbar=True, use_pyplot=False,
)
fig.suptitle(f"Captum Integrated Gradients — {pred_class_name}")
plt.tight_layout(); plt.show()

# ── Noise Tunnel (SmoothGrad) around IG ───────────────────────────────────────
nt_ig = NoiseTunnel(ig_captum)
# Averages IG over 10 noisy input versions — reduces gradient noise
nt_attr = nt_ig.attribute(
    input_tensor,
    target       = pred_class_idx,
    n_steps      = 20,
    nt_samples   = 10,   # number of noise samples to average over
    stdevs       = 0.1,  # noise std relative to input range
    nt_type      = "smoothgrad",
)

fig, axes = viz.visualize_image_attr_multiple(
    nt_attr[0].permute(1,2,0).cpu().detach().numpy(),
    denorm(input_tensor[0]),
    ["original_image", "heat_map"],
    ["all", "absolute_value"],
    ["Input", "SmoothGrad (IG + Noise Tunnel)"],
    show_colorbar=True, use_pyplot=False,
)
fig.suptitle(f"SmoothGrad — {pred_class_name}")
plt.tight_layout(); plt.show()

# 6) Occlusion Sensitivity

**A model-agnostic, gradient-free method**

Occlusion sensitivity slides a grey patch across the image, recording how much
the class score drops when each patch is blocked. High drop = the patch was important.

This is computationally expensive (one forward pass per patch position) but:
- Works for any model (no gradients needed — useful for non-differentiable models)
- Directly interpretable: we literally hide the feature and measure the effect
- Useful sanity check against gradient-based methods

In [ ]:
def occlusion_sensitivity(
    model: nn.Module,
    input_tensor: torch.Tensor,
    class_idx: int,
    patch_size: int = 32,
    stride: int     = 16,
    baseline_val: float = 0.0,
) -> np.ndarray:
    """
    Computes occlusion sensitivity map by sliding a grey patch over the image.

    For each patch position, the score drop when the patch is applied
    becomes the attribution for that region.

    Args:
        model         : Pretrained classifier (no gradients needed)
        input_tensor  : (1, 3, H, W) preprocessed input
        class_idx     : Target class to explain
        patch_size    : Side length of the occlusion patch (square)
        stride        : Step size of the sliding patch
        baseline_val  : Value to fill the occluded patch (0 = black in normalised space)

    Returns:
        sensitivity : (H, W) attribution map in [0, 1]
    """
    model.eval()
    _, _, H, W = input_tensor.shape

    # Baseline prediction score (no occlusion)
    with torch.no_grad():
        # (1, 3, H, W) → scalar
        base_score = F.softmax(model(input_tensor), dim=1)[0, class_idx].item()

    # Accumulate importance scores over occluded positions
    # (H, W)
    score_map = np.zeros((H, W))
    count_map = np.zeros((H, W))  # how many patches covered each pixel

    h_pos = range(0, H - patch_size + 1, stride)
    w_pos = range(0, W - patch_size + 1, stride)

    for h_start in h_pos:
        for w_start in w_pos:
            # Create an occluded copy
            # (1, 3, H, W)
            x_occ = input_tensor.clone()
            x_occ[:, :, h_start:h_start+patch_size, w_start:w_start+patch_size] = baseline_val

            with torch.no_grad():
                # Score drop: how much worse is the prediction with this patch occluded?
                occ_score  = F.softmax(model(x_occ), dim=1)[0, class_idx].item()
                score_drop = base_score - occ_score  # positive = region was important

            score_map[h_start:h_start+patch_size, w_start:w_start+patch_size] += score_drop
            count_map[h_start:h_start+patch_size, w_start:w_start+patch_size] += 1

    # Average over overlapping patches
    sensitivity = score_map / (count_map + 1e-8)

    # Normalise to [0, 1]
    sensitivity = np.clip(sensitivity, 0, None)  # only care about score drops
    sensitivity = (sensitivity - sensitivity.min()) / (sensitivity.max() - sensitivity.min() + 1e-8)
    return sensitivity


print("Computing occlusion sensitivity (this may take ~30s)...")
occ_map = occlusion_sensitivity(model, input_tensor, pred_class_idx, patch_size=32, stride=16)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(img_array); axes[0].set_title("Input"); axes[0].axis("off")
axes[1].imshow(overlay_cam(img_array, cam_map));  axes[1].set_title("GradCAM");            axes[1].axis("off")
axes[2].imshow(overlay_cam(img_array, occ_map));  axes[2].set_title("Occlusion Sensitivity"); axes[2].axis("off")
plt.suptitle(f"GradCAM vs Occlusion — {pred_class_name}")
plt.tight_layout(); plt.show()

# 7) Attribution Method Comparison and Sanity Checks

**The faithfulness problem**

A heatmap may *look* plausible while being completely unfaithful to how the model
actually makes decisions. Several sanity checks exist:

1. **Model parameter randomisation test** (Adebayo et al., 2018):
   Progressively randomise model weights. Faithful methods should produce
   completely different attributions as the model changes.
   Gradient-independent methods (e.g., edge detectors dressed as saliency maps) will not.

2. **Input invariance test**:
   Shift the input image by a constant value. The attribution should shift accordingly
   if the method is truly measuring input sensitivity.

3. **Completeness / conservation**:
   Sum of attributions should equal the model output difference (IG satisfies this by construction).

4. **Sensitivity** (axiomatic):
   If two inputs differ only in one feature and their outputs differ, that feature
   must receive non-zero attribution.

In [ ]:
def randomisation_test(
    model: nn.Module,
    input_tensor: torch.Tensor,
    class_idx: int,
) -> None:
    """
    Demonstrates the model parameter randomisation test for GradCAM.

    Progressively randomises weights from output to input layer.
    A faithful attribution method should produce noise-like maps
    when the model is randomised, because the model is no longer
    making meaningful predictions.
    """
    cam_maps = {}

    for label, randomise_from in [("Original", None), ("Last 2 layers randomised", 2)]:
        # Clone the model and randomise if requested
        test_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        test_model.eval()

        if randomise_from is not None:
            # Randomise the last `randomise_from` layer groups
            layers = list(test_model.children())
            for layer in layers[-randomise_from:]:
                for param in layer.parameters():
                    nn.init.normal_(param, mean=0.0, std=0.01)

        tgt_layer = test_model.layer4[-1].conv3
        gc        = GradCAM(test_model, tgt_layer)
        cam_m     = gc(input_tensor.clone().detach().requires_grad_(True), class_idx)
        cam_maps[label] = cam_m
        gc.remove_hooks()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_array); axes[0].set_title("Input"); axes[0].axis("off")
    for ax, (label, cam) in zip(axes[1:], cam_maps.items()):
        ax.imshow(overlay_cam(img_array, cam)); ax.set_title(label, fontsize=9); ax.axis("off")
    plt.suptitle("Randomisation test — faithful maps degrade when model is randomised")
    plt.tight_layout(); plt.show()


randomisation_test(model, input_tensor, pred_class_idx)

# ── All methods side-by-side ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (title, hmap) in zip(axes, [
    ("Input",       None),
    ("Saliency",    saliency_map),
    ("GradCAM",     cam_map),
    ("GradCAM++",   cam_pp),
    ("Int. Grads",  ig_map),
]):
    if hmap is None:
        ax.imshow(img_array)
    else:
        ax.imshow(overlay_cam(img_array, hmap))
    ax.set_title(title, fontsize=10); ax.axis("off")
plt.suptitle(f"All attribution methods — {pred_class_name}")
plt.tight_layout(); plt.show()

# Summary

| Method | Gradient | Axioms satisfied | Cost | Best for |
|---|---|---|---|---|
| Vanilla Saliency | Input grad | None | 1 backward pass | Quick debug |
| GradCAM | Layer grad | None | 1 backward pass | Spatial localisation |
| GradCAM++ | Layer grad (weighted) | None | 1 backward pass | Multiple objects |
| Integrated Gradients | Path integral | Completeness, Sensitivity | n_steps backward passes | Faithful attribution |
| Occlusion | None | Sensitivity | H×W/stride² forward passes | Any model, no gradients |

**Choosing an attribution method**

```
Need spatial localisation of the decision region?
  → GradCAM or GradCAM++

Need faithful pixel-level attribution with theoretical guarantees?
  → Integrated Gradients (use Captum for efficiency)

Model is non-differentiable (tree ensemble, API-only model)?
  → Occlusion or KernelSHAP

Publishing in a fairness or regulatory context?
  → Integrated Gradients + SmoothGrad + always run the randomisation sanity check
```

**Critical reading for PhD research**
- Adebayo et al. (2018) "Sanity Checks for Saliency Maps" — many popular
  methods fail basic sanity checks; read this before choosing a method.
- Kindermans et al. (2019) "The (Un)reliability of Saliency Methods" — input
  invariance is violated by many gradient methods.
- Hooker et al. (2019) "A Benchmark for Interpretability Methods" (ROAR/KAR) —
  the only evaluation framework that tests faithfulness through model retraining.